# Partial Cross-Entropy Loss for Weakly-Supervised Remote Sensing Segmentation

---

## TL;DR

Standard segmentation requires dense pixel masks. This notebook implements **Partial Cross-Entropy (pCE)** — a custom loss that trains on sparse point annotations only. The `MASK_labeled` binary tensor zeroes out gradient at unlabeled pixels:

$$pCE = \frac{\sum FocalLoss(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

Two experiments explore:
- **Experiment A**: How does point annotation density affect segmentation mIoU?
- **Experiment B**: How does focal loss gamma (hard example mining) interact with sparse supervision?

**Dataset**: ISPRS Potsdam (6-class urban remote sensing, free public benchmark)  
**Model**: DeepLabV3+ (ResNet-50, ImageNet pretrained)  
**Framework**: PyTorch + segmentation-models-pytorch

---

### Notebook structure
```
Cell 0  — Environment check & reproducibility seeds
Cell 1  — Imports & configuration
Cell 2  — Dataset download & tile extraction
Cell 3  — PotsdamDataset class + point sampler
Cell 4  — PartialCrossEntropyLoss implementation
Cell 5  — Model definition (DeepLabV3+)
Cell 6  — Training loop + metrics
Cell 7  — Experiment A: point density ablation
Cell 8  — Experiment B: focal gamma ablation
Cell 9  — Qualitative results visualization
Cell 10 — Results summary table & per-class IoU
```


In [ ]:
# ============================================================
# Cell 0 — Environment check & reproducibility
# ============================================================
import sys, torch, torchvision

print(f'Python:      {sys.version.split()[0]}')
print(f'PyTorch:     {torch.__version__}')
print(f'torchvision: {torchvision.__version__}')
print(f'CUDA:        {torch.version.cuda}')
print(f'Device:      {"cuda" if torch.cuda.is_available() else "cpu"}')
if torch.cuda.is_available():
    print(f'GPU:         {torch.cuda.get_device_name(0)}')

import random, numpy as np, os
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f'Seed {SEED} set for reproducibility.')


In [ ]:
# ============================================================
# Cell 1 — Imports & configuration
# ============================================================
# Uncomment if running in Colab / fresh environment:
# !pip install segmentation-models-pytorch -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix

# -----------------------------------------------------------
# Central configuration
# -----------------------------------------------------------
CFG = {
    'data_root':       './data/potsdam',
    'img_size':        512,
    'num_classes':     6,
    'train_ratio':     0.7,
    'backbone':        'resnet50',
    'batch_size':      4,
    'epochs':          30,
    'lr':              1e-4,
    'weight_decay':    1e-4,
    'grad_clip':       1.0,
    'device':          'cuda' if torch.cuda.is_available() else 'cpu',
    # Experiment A
    'point_densities': [0.001, 0.005, 0.01, 0.05],
    # Experiment B
    'focal_gammas':    [0.0, 0.5, 1.0, 2.0],
    'fixed_density':   0.01,
    'figures_dir':     './figures',
    'checkpoints_dir': './checkpoints',
}

CLASS_NAMES = [
    'Impervious surface', 'Building', 'Low vegetation',
    'Tree', 'Car', 'Clutter'
]
CLASS_COLORS_RGB = [
    (255, 255, 255),  # 0 Impervious surface (white)
    (0,   0,   255),  # 1 Building           (blue)
    (0,   255, 255),  # 2 Low vegetation     (cyan)
    (0,   255, 0  ),  # 3 Tree               (green)
    (255, 255, 0  ),  # 4 Car                (yellow)
    (255, 0,   0  ),  # 5 Clutter            (red)
]

for d in [CFG['figures_dir'], CFG['checkpoints_dir'],
          CFG['data_root'] + '/images', CFG['data_root'] + '/masks']:
    os.makedirs(d, exist_ok=True)

print('Configuration loaded.')
print(f'Device: {CFG["device"]}')
print(f'Exp A densities: {CFG["point_densities"]}')
print(f'Exp B gammas:    {CFG["focal_gammas"]}')


## Dataset Setup

### Dataset Choice 
ISPRS Potsdam or Vaihingen (both are free, public, and purpose-built for remote sensing segmentation).

- Potsdam: 38 patches, 6000×6000px each, 6 classes (impervious surface, building, low vegetation, tree, car, clutter), RGB + IR
- Vaihingen: 33 patches, varying sizes, same 6 classes, IR-R-G channels

### Option A — Download ISPRS Potsdam manually (Applied in this pipeline)
#### Data Source - Urban Modelling and Semantic Labelling Benchmark (UrbanSemLab) 
Full reference data (including all labels)
- Main benchmark overview: https://www.isprs.org/resources/datasets/benchmarks/UrbanSemLab/
- Potsdam-specific page: https://www.isprs.org/resources/datasets/benchmarks/UrbanSemLab/2d-sem-label-potsdam.aspx

#### Official Download

1. Go to the seafile server folder for Potsdam:
https://seafile.projekt.uni-hannover.de/f/429be50cc79d423ab6c4/
2. Password: CjwcipT4-P8g
3. Inside the folder you will find large zip files including:
    2_Ortho_RGB.zip (RGB images)
    5_Labels_all_noBoundary.zip

**Note**: The files are huge (13.3 GB total for the Potsdam folder). Download one zip at a time. After unzipping you’ll get the classic 38 patches (6000×6000 px each) with RGB orthophotos and the corresponding label TIFFs.

### Easier Mirror Download (Faster & More Convenient)
1. Google Drive - https://drive.google.com/drive/folders/1w3EJuyUGet6_qmLwGAWZ9vw5ogeG0zLz?usp=sharing (Public folder, direct download, maintained by MMSegmentation team)

### Quick Setup After Download
Once you have the two zips:

- Extract images 2_Ortho_RGB.zip → `data/potsdam/images/`
- Extract masks 5_Labels_all_noBoundary.zip → `data/potsdam/masks/`

Then you can follow the standard cropping script (MMSegmentation style) to split into 512×512 or 1024×1024 tiles:
Example from MMSegmentation (after placing zips in a folder)
`python tools/dataset_converters/potsdam.py /path/to/potsdam`

This will generate ~3,456 training tiles and ~2,016 validation tiles automatically.

### Vaihingen alternative (smaller, IR-R-G only):
Same seafile folder structure → use the Vaihingen link on the same page (https://seafile.projekt.uni-hannover.de/f/6a06a837b1f349cfa749/), same password.

### Option B — Synthetic data (instant, no download, for pipeline testing only)
Set `USE_SYNTHETIC = True` in the Dataset preparation cell below. Results will be random — synthetic only.

### Option C — Google Colab
Mount Drive and point `CFG['data_root']` at your folder.


In [ ]:
# ============================================================
# Cell 2 — Dataset preparation
# ============================================================
USE_SYNTHETIC = False   # Set True to use random synthetic tiles

def rgb_to_class(rgb_mask: np.ndarray) -> np.ndarray:
    """Convert Potsdam RGB annotation mask to integer class index map."""
    class_map = np.full(rgb_mask.shape[:2], fill_value=5, dtype=np.int64)
    for cls_idx, color in enumerate(CLASS_COLORS_RGB):
        match = np.all(rgb_mask == np.array(color), axis=-1)
        class_map[match] = cls_idx
    return class_map

def make_synthetic_dataset(root, n_tiles=30, tile_size=512, n_classes=6):
    """Generate random tiles for pipeline validation only."""
    img_dir  = Path(root) / 'images'
    mask_dir = Path(root) / 'masks'
    img_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    if len(list(img_dir.glob('*.png'))) >= n_tiles:
        print(f'Synthetic dataset already exists. Skipping.')
        return
    print(f'Generating {n_tiles} synthetic {tile_size}x{tile_size} tiles...')
    for i in range(n_tiles):
        img = np.random.randint(0, 255, (tile_size, tile_size, 3), dtype=np.uint8)
        Image.fromarray(img).save(img_dir / f'tile_{i:04d}.png')
        cls_idx = np.random.randint(0, n_classes, (tile_size, tile_size))
        rgb_m = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
        for c, col in enumerate(CLASS_COLORS_RGB):
            rgb_m[cls_idx == c] = col
        Image.fromarray(rgb_m).save(mask_dir / f'tile_{i:04d}.png')
    print('Done.')

def get_file_paths(root):
    img_dir  = Path(root) / 'images'
    mask_dir = Path(root) / 'masks'
    img_paths = sorted(img_dir.glob('*.png')) + sorted(img_dir.glob('*.tif'))
    pairs = [(str(i), str(mask_dir / i.name))
             for i in img_paths if (mask_dir / i.name).exists()]
    print(f'Found {len(pairs)} image/mask pairs in {root}')
    return pairs

def train_val_split(pairs, train_ratio=0.7, seed=42):
    rng = random.Random(seed)
    pairs = list(pairs)
    rng.shuffle(pairs)
    n = max(1, int(len(pairs) * train_ratio))
    return pairs[:n], pairs[n:]

if USE_SYNTHETIC:
    make_synthetic_dataset(CFG['data_root'])
else:
    n = len(list((Path(CFG['data_root']) / 'images').glob('*.png')))
    if n == 0:
        print('No images found. Set USE_SYNTHETIC=True or download ISPRS Potsdam.')
    else:
        print(f'Found {n} real images.')

pairs = get_file_paths(CFG['data_root'])
train_pairs, val_pairs = train_val_split(pairs, CFG['train_ratio'])
train_img_paths  = [p[0] for p in train_pairs]
train_mask_paths = [p[1] for p in train_pairs]
val_img_paths    = [p[0] for p in val_pairs]
val_mask_paths   = [p[1] for p in val_pairs]
print(f'Train: {len(train_pairs)} tiles | Val: {len(val_pairs)} tiles')


In [ ]:
# ============================================================
# Cell 3 — PotsdamDataset + stratified point sampler
# ============================================================

class PotsdamDataset(Dataset):
    """
    ISPRS Potsdam remote sensing segmentation dataset.
    Returns (image, full_mask, point_mask) per item.
    Point mask simulates sparse annotation via stratified sampling.
    """
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, image_paths, mask_paths, img_size=512,
                 point_density=0.01, augment=False):
        assert len(image_paths) == len(mask_paths)
        self.image_paths   = image_paths
        self.mask_paths    = mask_paths
        self.img_size      = img_size
        self.point_density = point_density
        self.augment       = augment
        self.normalize     = T.Normalize(mean=self.MEAN, std=self.STD)

    def __len__(self):
        return len(self.image_paths)

    def _load_and_crop(self, img_path, mask_path):
        img  = np.array(Image.open(img_path).convert('RGB'))
        mask = np.array(Image.open(mask_path).convert('RGB'))
        H, W = img.shape[:2]
        S = self.img_size
        # Pad if smaller than target
        if H < S or W < S:
            img  = np.pad(img,  ((0,max(0,S-H)),(0,max(0,S-W)),(0,0)), mode='reflect')
            mask = np.pad(mask, ((0,max(0,S-H)),(0,max(0,S-W)),(0,0)), mode='reflect')
            H, W = img.shape[:2]
        # Random crop for train, center crop for val
        r = random.randint(0, H-S) if self.augment else (H-S)//2
        c = random.randint(0, W-S) if self.augment else (W-S)//2
        return img[r:r+S, c:c+S], mask[r:r+S, c:c+S]

    def _augment(self, img, mask):
        if random.random() > 0.5:
            img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
        if random.random() > 0.5:
            img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
        return img, mask

    def sample_points(self, class_mask: np.ndarray) -> np.ndarray:
        """
        Stratified random point sampling — simulates sparse point annotation.
        Guarantees at least one sample per class present in the tile.
        Re-sampled every epoch as implicit augmentation.
        """
        H, W    = class_mask.shape
        n_total = max(1, int(H * W * self.point_density))
        classes = np.unique(class_mask)
        n_per   = max(1, n_total // len(classes))
        mask    = np.zeros((H, W), dtype=np.float32)
        for cls in classes:
            px = np.argwhere(class_mask == cls)
            n  = min(n_per, len(px))
            chosen = px[np.random.choice(len(px), n, replace=False)]
            mask[chosen[:, 0], chosen[:, 1]] = 1.0
        return mask

    def __getitem__(self, idx):
        img_np, mask_rgb = self._load_and_crop(
            self.image_paths[idx], self.mask_paths[idx]
        )
        if self.augment:
            img_np, mask_rgb = self._augment(img_np, mask_rgb)
        class_mask  = rgb_to_class(mask_rgb)
        point_mask  = self.sample_points(class_mask)
        img_t       = torch.from_numpy(img_np).permute(2,0,1).float() / 255.0
        img_t       = self.normalize(img_t)
        return (
            img_t,
            torch.from_numpy(class_mask).long(),
            torch.from_numpy(point_mask).float()
        )

# Smoke test
_ds = PotsdamDataset(
    train_img_paths[:2], train_mask_paths[:2],
    img_size=CFG['img_size'], point_density=0.01
)
_img, _mask, _pm = _ds[0]
print(f'Image:      {_img.shape}  {_img.dtype}')
print(f'Mask:       {_mask.shape}  {_mask.dtype}  classes={_mask.unique().tolist()}')
print(f'Point mask: {_pm.shape}  labeled={int(_pm.sum())} px ({_pm.mean()*100:.2f}%)')


## Cell 4 — Partial Cross-Entropy Loss (core deliverable)

The formula from the MeritiInc brief:

$$pCE = \frac{\sum FocalLoss(pred, GT) \times MASK_{labeled}}{\sum MASK_{labeled}}$$

Three design properties:
1. **Gradient isolation** — `pixel_loss × point_mask` zeroes gradient at unlabeled pixels
2. **Magnitude stability** — division by `Σ(MASK_labeled)` keeps loss scale constant regardless of sparsity
3. **Hard example focus** — focal weight `(1-pt)^γ` concentrates the sparse gradient budget on hard pixels


In [ ]:
# ============================================================
# Cell 4 — PartialCrossEntropyLoss  *** CORE DELIVERABLE ***
# ============================================================

class PartialCrossEntropyLoss(nn.Module):
    """
    Partial Cross-Entropy Loss for weakly-supervised semantic segmentation.

    Trains on sparse point annotations by computing loss ONLY at labeled pixels.
    Unlabeled pixels multiply by zero — contributing no gradient.

    Formula (MeritiInc brief):
        pCE = Σ(FocalLoss(pred, GT) × MASK_labeled) / Σ(MASK_labeled)

    Args:
        gamma (float):       Focal focusing parameter >= 0. 0 = standard CE.
        ignore_index (int):  Class index to exclude (e.g. boundary void pixels).
    """
    def __init__(self, gamma: float = 2.0, ignore_index: int = 255):
        super().__init__()
        self.gamma        = gamma
        self.ignore_index = ignore_index

    def focal_loss_per_pixel(self, logits, targets):
        """
        Element-wise focal loss (no reduction).

        Steps:
          1. CE per pixel = -log(p_t)
          2. p_t = exp(-CE) = model probability of the correct class
          3. focal_weight = (1 - p_t)^gamma
             -- p_t near 1 (easy) --> weight near 0 (down-weighted)
             -- p_t near 0 (hard) --> weight near 1 (full signal)
          4. focal = focal_weight * CE
        """
        # Step 1: per-pixel CE, shape (B, H, W)
        ce = F.cross_entropy(
            logits, targets,
            ignore_index=self.ignore_index,
            reduction='none'
        )
        if self.gamma == 0.0:
            return ce  # Exact standard CE; skip exp() for speed

        # Steps 2-4: focal weighting
        pt           = torch.exp(-ce)               # probability of correct class
        focal_weight = (1.0 - pt) ** self.gamma
        return focal_weight * ce                    # (B, H, W)

    def forward(self, logits, targets, point_mask):
        """
        Args:
            logits:     (B, C, H, W) raw model outputs (pre-softmax)
            targets:    (B, H, W)    integer ground-truth labels [0, C-1]
            point_mask: (B, H, W)    float32 binary; 1=labeled, 0=unlabeled
        Returns:
            Scalar differentiable loss
        """
        pixel_loss  = self.focal_loss_per_pixel(logits, targets)  # (B, H, W)
        masked_loss = pixel_loss * point_mask                      # zeros at unlabeled
        n_labeled   = point_mask.sum().clamp(min=1.0)             # safe denominator
        return masked_loss.sum() / n_labeled

    def __repr__(self):
        return f'PartialCrossEntropyLoss(gamma={self.gamma})'


# ---- Mathematical validation ----
print('Validating PartialCrossEntropyLoss...')
_B, _C, _H, _W = 2, 6, 64, 64
torch.manual_seed(42)
_lg = torch.randn(_B, _C, _H, _W)
_tg = torch.randint(0, _C, (_B, _H, _W))
_dn = torch.ones(_B, _H, _W)

# 1. gamma=0 with dense mask == standard CE
_pce0 = PartialCrossEntropyLoss(gamma=0.0)(_lg, _tg, _dn)
_sce  = F.cross_entropy(_lg, _tg)
print(f'  gamma=0 dense pCE: {_pce0.item():.5f}')
print(f'  Standard CE:       {_sce.item():.5f}')
print(f'  Match: {torch.allclose(_pce0, _sce, atol=1e-4)}')

# 2. Focal loss < CE for easy (confident) examples
_el = torch.full((_B,_C,_H,_W), -10.0); _el[:,0] = 10.0
_et = torch.zeros(_B,_H,_W, dtype=torch.long)
_ce_easy = PartialCrossEntropyLoss(gamma=0.0)(_el, _et, _dn).item()
_fl_easy = PartialCrossEntropyLoss(gamma=2.0)(_el, _et, _dn).item()
print(f'  Easy CE: {_ce_easy:.6f}  Focal(g=2): {_fl_easy:.6f}  focal<CE: {_fl_easy < _ce_easy}')

# 3. Empty mask -> loss = 0 (not NaN)
_em_loss = PartialCrossEntropyLoss(gamma=2.0)(_lg, _tg, torch.zeros(_B,_H,_W))
print(f'  Empty mask loss: {_em_loss.item():.4f} (should be 0.0, not NaN)')
print('All validations passed.')


In [ ]:
# ============================================================
# Cell 5 — Model: DeepLabV3+ with ResNet-50
# ============================================================

def build_model(num_classes=6, backbone='resnet50'):
    """
    DeepLabV3+ with ImageNet-pretrained ResNet-50 encoder.

    Rationale:
    - ASPP (Atrous Spatial Pyramid Pooling) captures multi-scale context:
      essential when objects range from cars (~10px) to buildings (~500px).
    - ImageNet pretraining compensates for sparse gradient signal of pCE.
    - activation=None returns raw logits; F.cross_entropy handles softmax
      internally for numerical stability.
    """
    return smp.DeepLabV3Plus(
        encoder_name    = backbone,
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = num_classes,
        activation      = None,
    )

_m = build_model(CFG['num_classes'], CFG['backbone'])
_x = torch.randn(1, 3, CFG['img_size'], CFG['img_size'])
_o = _m(_x)
print(f'Input:  {_x.shape}')
print(f'Output: {_o.shape}  (B, C, H, W logits)')
total = sum(p.numel() for p in _m.parameters())
train = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {train:,}')
del _m, _x, _o


In [ ]:
# ============================================================
# Cell 6 — Training loop + metrics
# ============================================================

def compute_metrics(preds, targets, num_classes):
    """
    Compute mIoU and pixel accuracy using full ground-truth masks.
    Evaluation always uses complete masks regardless of training sparsity.
    """
    preds_np   = preds.cpu().numpy().flatten().astype(np.int64)
    targets_np = targets.cpu().numpy().flatten().astype(np.int64)
    cm         = confusion_matrix(targets_np, preds_np, labels=list(range(num_classes)))
    inter      = np.diag(cm)
    union      = cm.sum(1) + cm.sum(0) - inter
    iou        = np.where(union > 0, inter / union, np.nan)
    return {
        'miou':          float(np.nanmean(iou)),
        'pixel_acc':     float(inter.sum() / (cm.sum() + 1e-8)),
        'iou_per_class': iou.tolist(),
    }


def train_one_run(model, train_loader, val_loader, loss_fn,
                  epochs, lr, device, run_name='run', verbose=True):
    """
    Full training + validation loop.
    Saves best checkpoint by val mIoU.
    Returns (history_dict, best_miou).
    """
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    history   = {'train_loss': [], 'val_miou': [], 'val_pixel_acc': [], 'lr': []}
    best_miou = 0.0
    ckpt_path = os.path.join(CFG['checkpoints_dir'], f'best_{run_name}.pth')

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        epoch_loss, n_batch = 0.0, 0
        for images, masks, point_masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            point_masks   = point_masks.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), masks, point_masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()
            epoch_loss += loss.item()
            n_batch    += 1
        scheduler.step()

        # --- Validate on FULL masks ---
        model.eval()
        all_preds, all_targets = [], []
        with torch.no_grad():
            for images, masks, _ in val_loader:
                preds = model(images.to(device)).argmax(dim=1).cpu()
                all_preds.append(preds)
                all_targets.append(masks)
        m = compute_metrics(torch.cat(all_preds), torch.cat(all_targets), CFG['num_classes'])

        avg_loss = epoch_loss / max(n_batch, 1)
        history['train_loss'].append(avg_loss)
        history['val_miou'].append(m['miou'])
        history['val_pixel_acc'].append(m['pixel_acc'])
        history['lr'].append(scheduler.get_last_lr()[0])

        if m['miou'] > best_miou:
            best_miou = m['miou']
            torch.save(model.state_dict(), ckpt_path)

        if verbose and (epoch + 1) % 5 == 0:
            print(f'  [{run_name}] ep{epoch+1:02d}/{epochs} '
                  f'loss={avg_loss:.4f} mIoU={m["miou"]:.4f} px_acc={m["pixel_acc"]:.4f}')

    print(f'  [{run_name}] Best mIoU={best_miou:.4f} saved -> {ckpt_path}')
    return history, best_miou

print('Training utilities ready.')


In [ ]:
# ============================================================
# Cell 7 — Experiment A: Point label density ablation
# ============================================================
# Purpose:   How does mIoU degrade as annotation becomes sparser?
# Hypothesis: Nonlinear degradation — robust above ~0.5%, sharp drop below it (rare classes absent from batches).
# Controlled: density in {0.001, 0.005, 0.01, 0.05}
# Fixed:      gamma = 2.0

print('=' * 60)
print('EXPERIMENT A: Point density vs. mIoU')
print('=' * 60)
density_results = {}
FIXED_GAMMA_A   = 2.0

for density in CFG['point_densities']:
    print(f'\n--- density={density*100:.1f}% | gamma={FIXED_GAMMA_A} ---')
    train_ds = PotsdamDataset(train_img_paths, train_mask_paths,
                               CFG['img_size'], density, augment=True)
    val_ds   = PotsdamDataset(val_img_paths, val_mask_paths,
                               CFG['img_size'], density, augment=False)
    tr_ld = DataLoader(train_ds, CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    va_ld = DataLoader(val_ds,   CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    torch.manual_seed(SEED)
    model  = build_model(CFG['num_classes'], CFG['backbone'])
    hist, best = train_one_run(
        model, tr_ld, va_ld, PartialCrossEntropyLoss(gamma=FIXED_GAMMA_A),
        CFG['epochs'], CFG['lr'], CFG['device'], f'expA_d{density}'
    )
    density_results[density] = {'history': hist, 'best_miou': best}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
densities = CFG['point_densities']
colors    = ['#534AB7','#1D9E75','#BA7517','#D85A30']
bests     = [density_results[d]['best_miou'] for d in densities]

axes[0].plot([d*100 for d in densities], bests, 'o-', color='#534AB7', lw=2, ms=8)
for d, m in zip(densities, bests):
    axes[0].annotate(f'{m:.3f}', (d*100, m), xytext=(0,8), textcoords='offset points',
                     ha='center', fontsize=9)
axes[0].set_xscale('log')
axes[0].set_xlabel('Label density (%)')
axes[0].set_ylabel('Best val mIoU')
axes[0].set_title('Exp A: Density vs. mIoU')
axes[0].grid(alpha=0.3)

for d, c in zip(densities, colors):
    axes[1].plot(density_results[d]['history']['val_miou'], color=c, lw=1.5, label=f'{d*100:.1f}%')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val mIoU')
axes[1].set_title('Exp A: mIoU curves'); axes[1].legend(title='Density'); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG["figures_dir"]}/experiment_a_density.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nExp A summary:')
for d, m in zip(densities, bests):
    print(f'  density={d*100:.1f}%  best_mIoU={m:.4f}')


In [ ]:
# ============================================================
# Cell 8 — Experiment B: Focal loss gamma ablation
# ============================================================
# Purpose:   Does focal hard-example mining help with sparse labels?
# Hypothesis: gamma in [1.0, 2.0] optimal. gamma=0 worst because easy background pixels dominate the sparse gradient budget.
# Controlled: gamma in {0.0, 0.5, 1.0, 2.0}
# Fixed:      density = 0.01 (1%)

print('=' * 60)
print('EXPERIMENT B: Focal gamma vs. mIoU')
print('=' * 60)
gamma_results   = {}
FIXED_DENSITY_B = CFG['fixed_density']

for gamma in CFG['focal_gammas']:
    print(f'\n--- gamma={gamma} | density={FIXED_DENSITY_B*100:.1f}% ---')
    train_ds = PotsdamDataset(train_img_paths, train_mask_paths,
                               CFG['img_size'], FIXED_DENSITY_B, augment=True)
    val_ds   = PotsdamDataset(val_img_paths, val_mask_paths,
                               CFG['img_size'], FIXED_DENSITY_B, augment=False)
    tr_ld = DataLoader(train_ds, CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    va_ld = DataLoader(val_ds,   CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    torch.manual_seed(SEED)
    model  = build_model(CFG['num_classes'], CFG['backbone'])
    hist, best = train_one_run(
        model, tr_ld, va_ld, PartialCrossEntropyLoss(gamma=gamma),
        CFG['epochs'], CFG['lr'], CFG['device'], f'expB_g{gamma}'
    )
    gamma_results[gamma] = {'history': hist, 'best_miou': best}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
gammas = CFG['focal_gammas']
colors = ['#B4B2A9','#9FE1CB','#5DCAA5','#0F6E56']
gmiou  = [gamma_results[g]['best_miou'] for g in gammas]

bars = axes[0].bar([f'gamma={g}' for g in gammas], gmiou,
                   color=colors, edgecolor='white', lw=0.5)
for bar, m in zip(bars, gmiou):
    axes[0].text(bar.get_x()+bar.get_width()/2, m+0.002, f'{m:.3f}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Focal gamma'); axes[0].set_ylabel('Best val mIoU')
axes[0].set_title('Exp B: Gamma vs. mIoU'); axes[0].grid(axis='y', alpha=0.3)

for g, c in zip(gammas, colors):
    axes[1].plot(gamma_results[g]['history']['val_miou'], color=c, lw=1.8, label=f'g={g}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val mIoU')
axes[1].set_title('Exp B: mIoU curves'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CFG["figures_dir"]}/experiment_b_gamma.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nExp B summary:')
for g, m in zip(gammas, gmiou):
    print(f'  gamma={g}  best_mIoU={m:.4f}')


In [ ]:
# ============================================================
# Cell 9 — Qualitative results visualization
# ============================================================

def class_to_rgb(class_map):
    rgb = np.zeros((*class_map.shape, 3), dtype=np.uint8)
    for i, c in enumerate(CLASS_COLORS_RGB):
        rgb[class_map == i] = c
    return rgb

def denormalize(t):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    return (t.permute(1,2,0).numpy() * std + mean).clip(0,1)

def visualize_predictions(model, dataset, device, n=4, seed=0):
    model.eval()
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), min(n, len(dataset)))
    fig, axes = plt.subplots(n, 4, figsize=(20, 5.5*n))
    if n == 1: axes = axes[np.newaxis,:]
    titles = ['Input','Point labels (red)','Prediction','Ground truth']
    for row, idx in enumerate(idxs):
        img_t, mask, pm = dataset[idx]
        with torch.no_grad():
            pred = model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
        img_np  = denormalize(img_t)
        pt_vis  = img_np.copy()
        pt_vis[pm.numpy() == 1] = [1.0, 0.0, 0.0]
        panels  = [img_np, pt_vis, class_to_rgb(pred)/255, class_to_rgb(mask.numpy())/255]
        for col, (panel, title) in enumerate(zip(panels, titles)):
            axes[row, col].imshow(panel); axes[row, col].axis('off')
            if row == 0: axes[row, col].set_title(title, fontsize=11)
    patches = [mpatches.Patch(color=[c/255 for c in col], label=n)
               for col, n in zip(CLASS_COLORS_RGB, CLASS_NAMES)]
    fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=10, bbox_to_anchor=(0.5,-0.02))
    plt.suptitle('Qualitative results — Partial CE Loss segmentation', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'{CFG["figures_dir"]}/qualitative_results.png', dpi=150, bbox_inches='tight')
    plt.show()

# Load best model from Experiment A
BEST_DENSITY = max(density_results, key=lambda d: density_results[d]['best_miou']) \
    if density_results else 0.01
ckpt = os.path.join(CFG['checkpoints_dir'], f'best_expA_d{BEST_DENSITY}.pth')
if os.path.exists(ckpt):
    bm = build_model(CFG['num_classes'], CFG['backbone'])
    bm.load_state_dict(torch.load(ckpt, map_location=CFG['device']))
    bm.to(CFG['device'])
    vis_ds = PotsdamDataset(val_img_paths, val_mask_paths, CFG['img_size'], BEST_DENSITY)
    visualize_predictions(bm, vis_ds, CFG['device'])
else:
    print(f'Checkpoint not found: {ckpt}. Run Experiment A first.')


In [ ]:
# ============================================================
# Cell 10 — Results summary table
# ============================================================

print('=' * 60)
print('FULL RESULTS SUMMARY')
print('=' * 60)

if density_results:
    print('\nExperiment A — Point Density (gamma=2.0)')
    print(f'  {"Density":>10} | {"Best mIoU":>10}')
    print('  ' + '-' * 25)
    for d in CFG['point_densities']:
        m = density_results.get(d, {}).get('best_miou', float('nan'))
        print(f'  {d*100:>9.1f}% | {m:>10.4f}')

if gamma_results:
    print(f'\nExperiment B — Focal Gamma (density={CFG["fixed_density"]*100:.1f}%)')
    print(f'  {"Gamma":>8} | {"Best mIoU":>10}')
    print('  ' + '-' * 22)
    for g in CFG['focal_gammas']:
        m = gamma_results.get(g, {}).get('best_miou', float('nan'))
        print(f'  {g:>8.1f} | {m:>10.4f}')

print('\nFigures saved to:', CFG['figures_dir'])
print('Checkpoints saved to:', CFG['checkpoints_dir'])
print('\nAll done!')
